<a href="https://colab.research.google.com/github/LuizaRamos/TOL506M_Final_Project/blob/main/notebooks/01_task1_scratch.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [4]:
# --- Colab setup for running notebooks with local package-style imports ---
#                 The code below, was generated by ChatGPT

# 1) Configure your repo details
REPO_URL = "https://github.com/LuizaRamos/TOL506M_Final_Project.git"
REPO_DIR = "/content/TOL506M_Final_Project"

# 2) Clone or update the repo
import os, sys, subprocess, pathlib

if not os.path.exists(REPO_DIR):
    subprocess.run(["git", "clone", REPO_URL, REPO_DIR], check=True)
else:
    # pull latest; safe if you want current main
    subprocess.run(["git", "-C", REPO_DIR, "pull", "--ff-only"], check=True)

# 3) Make the repo root the working directory
os.chdir(REPO_DIR)

# 4) Put the repo on PYTHONPATH so `from utils import ...` etc. works
if REPO_DIR not in sys.path:
    sys.path.insert(0, REPO_DIR)
os.environ["PYTHONPATH"] = REPO_DIR + os.pathsep + os.environ.get("PYTHONPATH", "")

In [5]:
import sys
import os
import subprocess
import importlib
import numpy as np
from pathlib import Path
from collections import Counter
import random
import matplotlib.pyplot as plt
import seaborn as sns
from PIL import Image
import torch

project_root = Path.cwd()
if project_root.name != "TOL506M_Final_Project":
    original = project_root
    while project_root.name != "TOL506M_Final_Project" and project_root != project_root.parent:
        project_root = project_root.parent

    if project_root.name == "TOL506M_Final_Project":
        os.chdir(project_root)
        print(f"Changed working directory from {original} to {project_root}")
    else:
        raise RuntimeError(
            "Could not locate the TOL506M_Final_Project root directory. "
            "Please run this notebook/script from within the project tree."
        )
else:
    print(f"Working directory: {project_root}")

if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

# Plot styling
sns.set_style("whitegrid")
plt.rcParams.update({"figure.figsize": (12, 6), "font.size": 12})

print(f"\nPyTorch version: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")

Working directory: /content/TOL506M_Final_Project

PyTorch version: 2.8.0+cu126
CUDA available: True


In [6]:
import kagglehub

# Download latest version
path = kagglehub.dataset_download("alessiocorrado99/animals10")

print(f'Dataset downloaded: {path}\n')

Dataset downloaded: /root/.cache/kagglehub/datasets/alessiocorrado99/animals10/versions/2



In [7]:
import json
import time
from typing import Dict, List, Tuple

import torch
import torch.nn as nn
from torchvision import datasets, transforms
import torch.optim as optim
from torch.optim.lr_scheduler import ReduceLROnPlateau, CosineAnnealingLR

from config import Config
from data.dataset import (
    WildlifeDataset,
    stratified_split,
    get_data_loaders,
    is_italian,
    translate_names,
    get_class_names
)
from models.resnet_scratch import ResNet18Scratch
from tasks.task1 import train_from_scratch
from utils.training import train_epoch, validate, EarlyStopping
from utils.evaluation import evaluate_model, get_confusion_matrix, compute_metrics
from utils.visualization import plot_training_curves, plot_confusion_matrix

data_fractions = Config.DATA_FRACTION
data_path = Path(path) / 'raw-img'
Config.DATA_PATH = data_path

basic_transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
])

full_dataset = datasets.ImageFolder(root=str(data_path), transform=basic_transform)
dataset = translate_names(full_dataset)
dataset

Dataset ImageFolder
    Number of datapoints: 26179
    Root location: /root/.cache/kagglehub/datasets/alessiocorrado99/animals10/versions/2/raw-img
    StandardTransform
Transform: Compose(
               Resize(size=(224, 224), interpolation=bilinear, max_size=None, antialias=True)
               ToTensor()
           )

In [8]:
train_indices, val_indices, test_indices = stratified_split(dataset,
                                                            train_size = 0.85,
                                                            val_size = 0.0,
                                                            test_size = 0.15,
                                                            random_seed = Config.RANDOM_SEED)

train_dataset = torch.utils.data.Subset(dataset, train_indices)
test_dataset = torch.utils.data.Subset(dataset, test_indices)

# Save the split indices
split_data = {
    'train_indices': train_indices.tolist() if isinstance(train_indices, torch.Tensor) else list(train_indices),
    'test_indices': test_indices.tolist() if isinstance(test_indices, torch.Tensor) else list(test_indices),
    'train_size': len(train_indices),
    'test_size': len(test_indices),
    'total_size': len(full_dataset),
    'split_ratio': '85/15'
}

split_save_path = Path(path) / 'train_test_split.json'
with open(split_save_path, 'w') as f:
    json.dump(split_data, f, indent=4)

print(f"Dataset split complete:")
print(f"  Training samples: {len(train_indices)} ({len(train_indices)/len(full_dataset)*100:.1f}%)")
print(f"  Testing samples: {len(test_indices)} ({len(test_indices)/len(full_dataset)*100:.1f}%)")
print(f"Split saved to: {split_save_path}")

TypeError: Object of type int64 is not JSON serializable